In [1]:
import os
import pandas as pd
import numpy as np
import gc

# datapath
datapath = 'D:/fypadpm/dataset/CSECICIDS2018/'

## Pre-Processing for Intrusion Detection Dataset

### Preprocessing on Train_validation and Test Datasets of Intrusion Detection

In [2]:
import pandas as pd
import numpy as np


def preprocess_train_val_df(df):
    """
    Optimized preprocessing for a single dataframe.
    No df.copy() used. All operations happen in-place.
    """

    # -------------------------------------------------------------
    # 1️⃣ Drop Date if exists
    # -------------------------------------------------------------
    if "Date" in df.columns:
        df.drop(columns="Date", inplace=True)

    # -------------------------------------------------------------
    # 2️⃣ Convert Timestamp
    # -------------------------------------------------------------
    if "Timestamp" in df.columns:
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")

    # -------------------------------------------------------------
    # 3️⃣ Convert non-excluded to numeric (vectorized)
    # -------------------------------------------------------------
    # exclude_cols = {"Timestamp", "Label", "Attack_Type", "Src IP", "Dst IP", "Flow ID"}
    # cols_to_convert = list(set(df.columns) - exclude_cols)

    # df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors="coerce")

    exclude_cols = ['Timestamp', 'Label', 'Attack_Type', 'Src IP', 'Dst IP', 'Flow ID']
    cols_to_convert = [col for col in df.columns if col not in exclude_cols]
    
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # -------------------------------------------------------------
    # 4️⃣ Remove negative Flow IAT Min / Max (single pass)
    # -------------------------------------------------------------
    if "Flow IAT Min" in df.columns:
        df = df[df["Flow IAT Min"] >= 0]
    if "Flow IAT Max" in df.columns:
        df = df[df["Flow IAT Max"] >= 0]

    # -------------------------------------------------------------
    # 5️⃣ Remove INF rows (across needed columns only)
    # -------------------------------------------------------------
    cols_inf = [c for c in ["Flow Byts/s", "Flow Pkts/s"] if c in df.columns]
    if cols_inf:
        df = df[~df[cols_inf].isin([np.inf, -np.inf]).any(axis=1)]

    # -------------------------------------------------------------
    # 6️⃣ Fill string columns in one call
    # -------------------------------------------------------------
    for col in ["Dst IP", "Src IP", "Flow ID"]:
        if col in df.columns:
            df[col].fillna("<<M>>", inplace=True)

    # -------------------------------------------------------------
    # 7️⃣ Fill Src Port
    # -------------------------------------------------------------
    if "Src Port" in df.columns:
        df["Src Port"].fillna(0, inplace=True)

    # -------------------------------------------------------------
    # 8️⃣ Drop NaN in critical columns
    # -------------------------------------------------------------
    critical = [c for c in ["Flow Byts/s", "Flow Pkts/s"] if c in df.columns]
    if critical:
        df.dropna(subset=critical, inplace=True)

    # -------------------------------------------------------------
    # 9️⃣ Drop all-zero numeric columns (vectorized)
    # -------------------------------------------------------------
    #numeric_cols = df.select_dtypes(include="number").columns
    #zero_cols = numeric_cols[(df[numeric_cols] == 0).all()]
    #if len(zero_cols) > 0:
    #    df.drop(columns=list(zero_cols), inplace=True)

    # -------------------------------------------------------------
    # 🔟 Final data integrity diagnostics
    # -------------------------------------------------------------
    numeric_cols = df.select_dtypes(include="number").columns

    neg_cols = (df[numeric_cols] < 0).sum()
    inf_cols = np.isinf(df[numeric_cols]).sum()
    null_cols = df.isnull().sum()

    print("Columns with negative values:")
    print(neg_cols[neg_cols > 0])

    print("\nColumns with inf values:")
    print(inf_cols[inf_cols > 0])

    print("\nColumns with null values:")
    print(null_cols[null_cols > 0])

    print("\nFinal shape:", df.shape)

    return df


In [3]:
import gc

chunks = []
for file in ["df_train_setm.csv", "df_validation_setm.csv"]:
    for chunk in pd.read_csv(datapath + file, low_memory=False, chunksize=5000000):
        chunk = preprocess_train_val_df(chunk)
        chunks.append(chunk)
        gc.collect()

df_combined = pd.concat(chunks, axis=0, ignore_index=True)
del chunks
gc.collect()

df_combined.to_csv(datapath + "processed_train_validation_set.csv", index=False)
del df_combined
gc.collect()

C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:21: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")
C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna("<<M>>", inplace=True)
C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:63: FutureWarn

Columns with negative values:
Init Fwd Win Byts    1581016
Init Bwd Win Byts    2540995
dtype: int64

Columns with inf values:
Series([], dtype: int64)

Columns with null values:
Series([], dtype: int64)

Final shape: (4964484, 85)


C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:21: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")
C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna("<<M>>", inplace=True)
C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:63: FutureWarn

Columns with negative values:
Init Fwd Win Byts    1395516
Init Bwd Win Byts    2594493
dtype: int64

Columns with inf values:
Series([], dtype: int64)

Columns with null values:
Series([], dtype: int64)

Final shape: (4970977, 85)


C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna("<<M>>", inplace=True)
C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:57: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<<M>>' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df[col].fillna("<<M>>", inplace=True)
C:\Users\loke\AppData\Local\Temp\ipykernel_10136\65

Columns with negative values:
Init Fwd Win Byts    170804
Init Bwd Win Byts    365280
dtype: int64

Columns with inf values:
Series([], dtype: int64)

Columns with null values:
Series([], dtype: int64)

Final shape: (755185, 85)


C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:21: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")
C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna("<<M>>", inplace=True)
C:\Users\loke\AppData\Local\Temp\ipykernel_10136\653886782.py:63: FutureWarn

Columns with negative values:
Init Fwd Win Byts     496587
Init Bwd Win Byts    1186481
dtype: int64

Columns with inf values:
Series([], dtype: int64)

Columns with null values:
Series([], dtype: int64)

Final shape: (2328308, 85)


0

In [4]:
df_head = pd.read_csv(datapath + "processed_train_validation_set.csv", nrows=5)
print("Columns:", df_head.shape[1])

with open(datapath + "processed_train_validation_set.csv", 'r') as f:
    row_count = sum(1 for line in f) - 1
print("Rows:", row_count)

Columns: 85
Rows: 13018954


In [8]:
import pandas as pd
import numpy as np

def preprocess_test_df(df):
    if 'Date' in df.columns:
        df = df.drop(columns='Date')

    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')

    exclude_cols = ['Timestamp', 'Label', 'Attack_Type', 'Src IP', 'Dst IP', 'Flow ID']
    cols_to_convert = [col for col in df.columns if col not in exclude_cols]
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in ['Dst IP', 'Src IP', 'Flow ID']:
        if col in df.columns:
            df[col] = df[col].fillna("<<M>>")
    for col in ['Src Port', 'Dst Port', 'Protocol']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 只处理 critical columns 的 inf 和 NaN
    critical = [c for c in ['Flow Byts/s', 'Flow Pkts/s'] if c in df.columns]
    if critical:
        df = df[~df[critical].isin([np.inf, -np.inf]).any(axis=1)]
        df = df.dropna(subset=critical)

    # Remove negative values (except allowed columns)
    allowed_negative_cols = ['Init Fwd Win Byts', 'Init Bwd Win Byts']
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    cols_to_check_neg = [col for col in numeric_cols if col not in allowed_negative_cols]
    df = df[(df[cols_to_check_neg] >= 0).all(axis=1)]

    # Final validation
    numeric_cols = df.select_dtypes(include=['number']).columns
    neg_counts = (df[numeric_cols] < 0).sum()
    inf_counts = np.isinf(df[numeric_cols]).sum()
    null_counts = df.isna().sum()
    print("Negative values:\n", neg_counts[neg_counts > 0])
    print("\nInf values:\n", inf_counts[inf_counts > 0])
    print("\nNull values:\n", null_counts[null_counts > 0])
    print("\nFinal shape:", df.shape)
    return df

In [9]:
df_test = pd.read_csv(datapath + "df_test_setm.csv", low_memory=False)
df_test = preprocess_test_df(df_test)
df_test.to_csv(datapath + "processed_test_set.csv", index=False)
del df_test
gc.collect()

C:\Users\loke\AppData\Local\Temp\ipykernel_10136\2989972712.py:9: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')


Negative values:
 Init Fwd Win Byts     788231
Init Bwd Win Byts    1412780
dtype: int64

Inf values:
 Series([], dtype: int64)

Null values:
 Series([], dtype: int64)

Final shape: (2719750, 85)


0

### Stratify split of train and validation Datasets

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
import gc

# Attack_Type 
at_col = pd.read_csv(datapath + "processed_train_validation_set.csv", usecols=["Attack_Type"])
print(at_col["Attack_Type"].value_counts())

# chunk
train_chunks = []
val_chunks = []

for chunk in pd.read_csv(datapath + "processed_train_validation_set.csv", low_memory=False, chunksize=5000000):
    c_train, c_val = train_test_split(chunk, test_size=0.2, random_state=42, stratify=chunk['Attack_Type'])
    train_chunks.append(c_train)
    val_chunks.append(c_val)
    gc.collect()

df_train = pd.concat(train_chunks, ignore_index=True)
del train_chunks
gc.collect()
df_train.to_csv(datapath + "processed_train_set.csv", index=False)
print(f"Train size: {len(df_train)}")
del df_train
gc.collect()

df_val = pd.concat(val_chunks, ignore_index=True)
del val_chunks
gc.collect()
df_val.to_csv(datapath + "processed_validation_set.csv", index=False)
print(f"Validation size: {len(df_val)}")
del df_val
gc.collect()

print("Done! Stratified train/val split saved.")

Attack_Type
Benign          10946848
DDoS             1155241
DoS               391936
Bot               236327
Brute Force       144656
Infiltration      143151
Web Attack           795
Name: count, dtype: int64
Train size: 10415163
Validation size: 2603791
Done! Stratified train/val split saved.


### Encode Label and  Attack_type 

In [11]:
import pandas as pd
import json

df_train=pd.read_csv(datapath+"processed_train_set.csv")
df_val=pd.read_csv(datapath+"processed_validation_set.csv")
df_test=pd.read_csv(datapath+"processed_test_set.csv")

# ----------------- Load mapping.json -----------------
with open(r"D:\fypadpm\proj1\mappings.json", "r") as f:
    mapping_data = json.load(f)

# Mappings from JSON (int → string)
label_mapping = mapping_data["label_mapping"]
attack_type_mapping = mapping_data["attack_type_mapping"]

# ----------------- Reverse mappings (string → int) -----------------
label_reverse_mapping = {v: int(k) for k, v in label_mapping.items()}
attack_type_reverse_mapping = {v: int(k) for k, v in attack_type_mapping.items()}

# ----------------- Apply mapping to df -----------------
# df must already be loaded before this

for df in [df_train, df_val, df_test]:
    df['Label'] = df['Label'].map(label_reverse_mapping).astype(int)
    df['Attack_Type'] = df['Attack_Type'].map(attack_type_reverse_mapping).astype(int)

# # ----------------- Save mappings (optional) -----------------
# joblib.dump(label_reverse_mapping, "D:/r6g/prog1/cicids2018M2/label_reverse_mapping.pkl")
# joblib.dump(attack_type_reverse_mapping, "D:/r6g/prog1/cicids2018M2/attack_type_reverse_mapping.pkl")

print("Label reverse mapping:", label_reverse_mapping)
print("Attack Type reverse mapping:", attack_type_reverse_mapping)

df_train.to_csv(datapath+'encoded_train_set.csv', index=False)
df_val.to_csv(datapath+'encoded_validation_set.csv', index=False)
df_test.to_csv(datapath+'encoded_test_set.csv', index=False)


Label reverse mapping: {'Benign': 0, 'Bot': 1, 'Brute Force -Web': 2, 'Brute Force -XSS': 3, 'DDOS attack-HOIC': 4, 'DDOS attack-LOIC-UDP': 5, 'DDoS attacks-LOIC-HTTP': 6, 'DoS attacks-GoldenEye': 7, 'DoS attacks-Hulk': 8, 'DoS attacks-SlowHTTPTest': 9, 'DoS attacks-Slowloris': 10, 'FTP-BruteForce': 11, 'Infilteration': 12, 'SQL Injection': 13, 'SSH-Bruteforce': 14}
Attack Type reverse mapping: {'Benign': 0, 'Bot': 1, 'Brute Force': 2, 'DDoS': 3, 'DoS': 4, 'Infiltration': 5, 'Web Attack': 6}


### Normalize Numerical Feature

In [3]:
from sklearn.preprocessing import MinMaxScaler
import joblib
import pandas as pd
import gc

exclude_cols = ['Attack_Type', 'Label']

# 1. 先读一小部分确定 feature columns
df_head = pd.read_csv(datapath + "encoded_train_set.csv", nrows=5)
feature_numeric_cols = (
    df_head.drop(columns=exclude_cols, errors='ignore')
           .select_dtypes(include=['number'])
           .columns.tolist()
)
joblib.dump(feature_numeric_cols, datapath + "feature_numeric_cols.pkl")

# 2. partial_fit scaler on train chunks
scaler = MinMaxScaler()
for chunk in pd.read_csv(datapath + "encoded_train_set.csv", low_memory=False, chunksize=500000):
    scaler.partial_fit(chunk[feature_numeric_cols])
    gc.collect()
joblib.dump(scaler, datapath + "minmax_scaler_on_feature_numeric_cols.pkl")
print("Scaler fitted.")

# 3. Transform and save train
with open(datapath + "normalized_train_set.csv", "w") as f:
    first = True
    for chunk in pd.read_csv(datapath + "encoded_train_set.csv", low_memory=False, chunksize=500000):
        chunk[feature_numeric_cols] = scaler.transform(chunk[feature_numeric_cols])
        chunk.to_csv(f, index=False, header=first)
        first = False
        gc.collect()
print("Train saved.")

# 4. Transform and save validation
with open(datapath + "normalized_validation_set.csv", "w") as f:
    first = True
    for chunk in pd.read_csv(datapath + "encoded_validation_set.csv", low_memory=False, chunksize=500000):
        chunk[feature_numeric_cols] = scaler.transform(chunk[feature_numeric_cols])
        chunk[feature_numeric_cols] = chunk[feature_numeric_cols].clip(lower=0, upper=1)
        chunk.to_csv(f, index=False, header=first)
        first = False
        gc.collect()
print("Validation saved.")

# 5. Transform and save test
with open(datapath + "normalized_test_set.csv", "w") as f:
    first = True
    for chunk in pd.read_csv(datapath + "encoded_test_set.csv", low_memory=False, chunksize=500000):
        chunk[feature_numeric_cols] = scaler.transform(chunk[feature_numeric_cols])
        chunk[feature_numeric_cols] = chunk[feature_numeric_cols].clip(lower=0, upper=1)
        chunk.to_csv(f, index=False, header=first)
        first = False
        gc.collect()
print("Test saved.")

print("Normalization completed and saved successfully.")

Scaler fitted.
Train saved.
Validation saved.
Test saved.
Normalization completed and saved successfully.


In [4]:
print(feature_numeric_cols)

['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'B

In [6]:
df_head = pd.read_csv(datapath + "normalized_train_set.csv", nrows=5)
df_head[feature_numeric_cols].head()

,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Src Port
0,0.051713,0.352941,1.898448e-02,0.000029,0.000057,2.748668e-05,1.011125e-05,0.010506,0.000000,0.007717,...,0.416667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.794919
1,0.000809,1.000000,8.408333e-06,0.000000,0.000008,9.577242e-07,6.459435e-07,0.000621,0.027397,0.002420,...,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.891722
2,0.006760,0.352941,7.400000e-06,0.000007,0.000000,1.843619e-06,0.000000e+00,0.000714,0.000000,0.001553,...,0.416667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.820996
3,0.001221,0.352941,7.833333e-07,0.000004,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,...,0.416667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.800702
4,0.000809,1.000000,1.707583e-04,0.000000,0.000008,1.221098e-06,5.052429e-07,0.000791,0.034932,0.003085,...,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.795911


In [11]:
# import numpy as np

# # Only numeric columns
# numeric_cols = df_train.select_dtypes(include=[np.number]).columns

# # Infinity mask
# inf_mask = np.isinf(df_train[numeric_cols])

# # Columns with inf
# cols_with_inf = inf_mask.any(axis=0)

# print("Columns with ±inf:")
# print(cols_with_inf[cols_with_inf].index.tolist())


<!-- ### Split train and validation Dataset Stratically -->

### Feature Selection

#### Feature Selection: Boruta

In [2]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import joblib

# --------------------------
# Load train dataset
# --------------------------
df_train = pd.read_csv(datapath+"normalized_train_set.csv")
X = df_train.drop(['Attack_Type', 'Label'], axis=1).select_dtypes(include=[np.number]).astype(np.float32)
y = df_train['Attack_Type']

# --------------------------
# Boruta feature selection
# --------------------------
rf = RandomForestClassifier(n_jobs=-1, max_depth=5, random_state=42)
boruta = BorutaPy(estimator=rf, n_estimators='auto', random_state=42)
boruta.fit(X.values, y.values)

# Selected features
selected_features = X.columns[boruta.support_].tolist()
print("Boruta selected features:", selected_features)

# Feature importance
orig_importances = boruta.estimator.feature_importances_[:len(X.columns)]
ranking_df = pd.DataFrame({
    'feature': X.columns,
    'importance': orig_importances,
    'selected_by_boruta': boruta.support_
}).sort_values(by='importance', ascending=False)

top_20 = ranking_df.head(20)
top_20_features = top_20['feature'].tolist()
print("Top 20 features:\n", top_20)

# Save top 20 features
joblib.dump(top_20_features, datapath+"t20f_boruta.pkl")

# --------------------------
# Build final datasets
# --------------------------
def save_boruta_dataset(df, features, out_path):
    df_final = df[features + ['Label', 'Attack_Type']]
    df_final.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return df_final

# Train
df_train_final = save_boruta_dataset(df_train, top_20_features, datapath+"Boruta_train.csv")

# Validation
df_val = pd.read_csv(datapath+"normalized_validation_set.csv")
save_boruta_dataset(df_val, top_20_features, datapath+"Boruta_validation.csv")

# Test
df_test = pd.read_csv(datapath+"normalized_test_set.csv")
save_boruta_dataset(df_test, top_20_features, datapath+"Boruta_test.csv")

# Preview
print(df_train_final.head())


Boruta selected features: ['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Ini

## TOP 10/20/30

In [4]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import joblib
import gc

# chunk 读取
chunks = []
for chunk in pd.read_csv(datapath + "normalized_train_set.csv", low_memory=False, chunksize=500000):
    chunk[chunk.select_dtypes(include=['number']).columns] = chunk.select_dtypes(include=['number']).astype(np.float32)
    chunks.append(chunk)
    gc.collect()
df_train = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

X = df_train.drop(['Attack_Type', 'Label'], axis=1).select_dtypes(include=[np.number])
y = df_train['Attack_Type']

rf = RandomForestClassifier(n_jobs=-1, max_depth=5, n_estimators=50, random_state=42)
boruta = BorutaPy(estimator=rf, n_estimators='auto', max_iter=50, random_state=42)
boruta.fit(X.values, y.values)

selected_features = X.columns[boruta.support_].tolist()
print("Boruta selected features:", selected_features)

orig_importances = boruta.estimator.feature_importances_[:len(X.columns)]
ranking_df = pd.DataFrame({
    'feature': X.columns,
    'importance': orig_importances,
    'selected_by_boruta': boruta.support_
}).sort_values(by='importance', ascending=False)

top_10_features = ranking_df.head(10)['feature'].tolist()
top_20_features = ranking_df.head(20)['feature'].tolist()
top_30_features = ranking_df.head(30)['feature'].tolist()
print("Top 10:", top_10_features)
print("Top 20:", top_20_features)
print("Top 30:", top_30_features)

all_feature_sets = {
    "top10": top_10_features,
    "top20": top_20_features,
    "top30": top_30_features
}
joblib.dump(all_feature_sets, datapath + "boruta_top_feature_sets.pkl")
print("Saved PKL: boruta_top_feature_sets.pkl")

del X, y, rf, boruta, df_train
gc.collect()

# Save datasets
for features, suffix in [(top_10_features, "t10"), (top_20_features, "t20"), (top_30_features, "t30")]:
    for in_file, label in [("normalized_train_set.csv", "train"), ("normalized_validation_set.csv", "validation"), ("normalized_test_set.csv", "test")]:
        with open(datapath + f"Boruta_{label}_{suffix}.csv", "w") as f:
            first = True
            for chunk in pd.read_csv(datapath + in_file, low_memory=False, chunksize=500000):
                chunk[features + ['Label', 'Attack_Type']].to_csv(f, index=False, header=first)
                first = False
                gc.collect()
        print(f"Saved: Boruta_{label}_{suffix}.csv")

print("All Boruta top-N datasets created successfully.")

Boruta selected features: ['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Ini

#### Feature Selection using CorrMI

In [5]:
import numpy as np
import pandas as pd
import joblib
import gc
from sklearn.feature_selection import mutual_info_classif

# chunk 读取
chunks = []
for chunk in pd.read_csv(datapath + "normalized_train_set.csv", low_memory=False, chunksize=5000000):
    chunk[chunk.select_dtypes(include=['number']).columns] = chunk.select_dtypes(include=['number']).astype(np.float32)
    chunks.append(chunk)
    gc.collect()
df_train = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

X = df_train.drop(['Attack_Type', 'Label'], axis=1).select_dtypes(include=[np.number])
y = df_train['Attack_Type']

# Correlation Filtering
corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]
X_filtered = X.drop(columns=to_drop)
del X, corr_matrix, upper_tri
gc.collect()

# Mutual Information
mi = mutual_info_classif(X_filtered, y, random_state=42)
mi_df = pd.DataFrame({'Feature': X_filtered.columns, 'MI_Score': mi})
mi_sorted = mi_df.sort_values(by="MI_Score", ascending=False)

top10_features = mi_sorted.head(10)['Feature'].tolist()
top20_features = mi_sorted.head(20)['Feature'].tolist()
top30_features = mi_sorted.head(30)['Feature'].tolist()
print("Top 10 MI features:", top10_features)
print("Top 20 MI features:", top20_features)
print("Top 30 MI features:", top30_features)

all_feature_sets = {
    "top10": top10_features,
    "top20": top20_features,
    "top30": top30_features
}
joblib.dump(all_feature_sets, datapath + "corrmi_feature_sets.pkl")
print("Saved: corrmi_feature_sets.pkl (top10/top20/top30)")

del X_filtered, y, mi, mi_df, mi_sorted, df_train
gc.collect()

# Save datasets
for features, suffix in [(top10_features, "t10"), (top20_features, "t20"), (top30_features, "t30")]:
    for in_file, label in [("normalized_train_set.csv", "train"), ("normalized_validation_set.csv", "validation"), ("normalized_test_set.csv", "test")]:
        with open(datapath + f"CorrMI_{label}_{suffix}.csv", "w") as f:
            first = True
            for chunk in pd.read_csv(datapath + in_file, low_memory=False, chunksize=500000):
                chunk[features + ['Label', 'Attack_Type']].to_csv(f, index=False, header=first)
                first = False
                gc.collect()
        print(f"Saved: CorrMI_{label}_{suffix}.csv")

print("All Corr+MI datasets generated successfully.")

Top 10 MI features: ['Init Fwd Win Byts', 'Dst Port', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Bwd Pkt Len Mean', 'Pkt Len Var', 'Flow IAT Max', 'Flow IAT Mean', 'Flow Pkts/s', 'Flow Duration']
Top 20 MI features: ['Init Fwd Win Byts', 'Dst Port', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Bwd Pkt Len Mean', 'Pkt Len Var', 'Flow IAT Max', 'Flow IAT Mean', 'Flow Pkts/s', 'Flow Duration', 'Bwd Pkts/s', 'Init Bwd Win Byts', 'Fwd Seg Size Min', 'Tot Bwd Pkts', 'Flow Byts/s', 'Tot Fwd Pkts', 'Flow IAT Std', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Protocol']
Top 30 MI features: ['Init Fwd Win Byts', 'Dst Port', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Bwd Pkt Len Mean', 'Pkt Len Var', 'Flow IAT Max', 'Flow IAT Mean', 'Flow Pkts/s', 'Flow Duration', 'Bwd Pkts/s', 'Init Bwd Win Byts', 'Fwd Seg Size Min', 'Tot Bwd Pkts', 'Flow Byts/s', 'Tot Fwd Pkts', 'Flow IAT Std', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Protocol', 'Bwd IAT Std', 'Down/Up Ratio', 'Src Port', 'Fwd Pkt Len Min', 'Bwd Pkt Len Min', 'RST Flag Cnt', 'Act

## Dataset Preparation for Mitigation

In [14]:
# --------------------------------------------------------------------------------------------------------
# Preprocessing:
#    1) From processed train_val & test datasets perform one-hot on Attack_Type to generate M1, M2,... M16
#    2) Stratify split train and validation datasets
#    3) Normalize Features
#    4) Feature selection for multi-label best 20 for all labels or union of best 20 from all labels
# -----------------------------------------------------------------------------------------------------------

### Step 1 - One-hot Encoding on Attack Types

In [16]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import json

# Load the mapping file
with open("D:/r6g/prog1/cicids2018M2/mappings.json", "r") as f:
    mapping_data = json.load(f)

# Extract the attack_type_mitigation_mapping
attack_type_mitigation_mapping = mapping_data.get("attack_type_mitigation_mapping", {})
attack_type_mitigation_mapping_int = {int(k): v for k, v in attack_type_mitigation_mapping.items()}
# Inspect the mapping
print(attack_type_mitigation_mapping_int)

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

def one_hot_encode_attack_types(df_corr, attack_type_mitigation_mapping):
    """
    Convert Attack_Type in df_corr to multilabel one-hot encoding
    based on attack_type_mitigation_mapping.

    Parameters:
    - df_corr: pd.DataFrame containing 'Attack_Type' column
    - attack_type_mitigation_mapping: dict mapping attack types to mitigation labels
    
    Returns:
    - df_final: original df_corr with one-hot encoded mitigation columns added
    """
    X = df_corr.drop(columns=['Attack_Type'])
    y = df_corr[['Attack_Type']]
    
    # Convert attack types to multilabel format
    y_labels = y['Attack_Type'].map(lambda x: attack_type_mitigation_mapping.get(x, []))
    
    ordered_labels = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16']
    
    # One-hot encode attack types
    mlb = MultiLabelBinarizer(classes=ordered_labels)
    y_transformed = mlb.fit_transform(y_labels)
    
    y_encoded_df = pd.DataFrame(y_transformed, columns=ordered_labels)
    df_final = pd.concat([X,y, y_encoded_df], axis=1)
    
    return df_final
   
# Load Processed Train_Val & Test Datasets
df_train_val=pd.read_csv(datapath+"processed_train_validation_set.csv")
df_test=pd.read_csv(datapath+"processed_test_set.csv")

# one-hot
df_train_val_oh = one_hot_encode_attack_types(df_train_val, attack_type_mitigation_mapping_int)
df_test_oh = one_hot_encode_attack_types(df_test, attack_type_mitigation_mapping_int)

df_train_val_oh.to_csv(datapath+'encoded_mitigation_train_validation.csv', index=False)
df_test_oh.to_csv(datapath+'encoded_mitigation_test.csv', index=False)

{1: ['M1', 'M2', 'M3', 'M8', 'M9', 'M10', 'M14', 'M15', 'M16'], 2: ['M2', 'M4', 'M5', 'M6', 'M7', 'M13', 'M14'], 3: ['M1', 'M3', 'M4', 'M5', 'M7', 'M8', 'M15', 'M16'], 4: ['M1', 'M3', 'M4', 'M5', 'M7', 'M8', 'M15', 'M16'], 5: ['M2', 'M4', 'M5', 'M6', 'M7', 'M13', 'M14'], 6: ['M1', 'M2', 'M6', 'M7', 'M9', 'M15', 'M16']}


### Step 2 - Stratify split train & validation datasets

In [17]:
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# -----------------------------
# Load datasets
# -----------------------------
df_combined = pd.read_csv(datapath+"encoded_mitigation_train_validation.csv")

# -----------------------------
# Define multilabel columns
# -----------------------------
ml_labels = ['M1','M2','M3','M4','M5','M6','M7','M8','M9', 'M10','M11','M12','M13','M14','M15','M16']

# -----------------------------
# Features (including Label and Attack_Type)
# -----------------------------
feature_cols = [c for c in df_combined.columns if c not in ml_labels]
X = df_combined[feature_cols]
y = df_combined[ml_labels]

# -----------------------------
# Multilabel Stratified Split
# -----------------------------
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, val_idx in msss.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

# -----------------------------
# Combine features + multilabels
# -----------------------------
df_train_split = pd.concat([X_train, y_train], axis=1)
df_val_split   = pd.concat([X_val, y_val], axis=1)

# -----------------------------
# Save final datasets
# -----------------------------
df_train_split.to_csv(datapath+'encoded_mitigation_train.csv', index=False)
df_val_split.to_csv(datapath+'encoded_mitigation_validation.csv', index=False)

print("Final split complete.")
print("Train shape:", df_train_split.shape)
print("Validation shape:", df_val_split.shape)

Final split complete.
Train shape: (10415285, 93)
Validation shape: (2603822, 93)


### Step 3 - Normalizing train, val and test datasets

In [18]:
from sklearn.preprocessing import MinMaxScaler
import joblib
import pandas as pd

# Load train set
df_train = pd.read_csv(datapath+"encoded_mitigation_train.csv")

ml_labels = ['M1','M2','M3','M4','M5','M6','M7','M8','M9', 'M10','M11','M12','M13','M14','M15','M16']

exclude_cols = ['Attack_Type', 'Label'] + ml_labels

# 1️⃣ Choose numeric feature columns ONLY (exclude targets)
feature_numeric_cols = (
    df_train.drop(columns=exclude_cols, errors='ignore')
            .select_dtypes(include=['number'])
            .columns.tolist()
)

# 2️⃣ Save numeric columns for future use
joblib.dump(feature_numeric_cols, datapath+"mitigation_feature_numeric_cols.pkl")

# 3️⃣ Scale numeric features in-place
scaler = MinMaxScaler()
df_train[feature_numeric_cols] = scaler.fit_transform(df_train[feature_numeric_cols])

# 4️⃣ Save the fitted scaler
joblib.dump(scaler, datapath+"minmax_scaler_on_mitigation_feature_numeric_cols.pkl")

# 5️⃣ Save the normalized train set
df_train.to_csv(datapath+'normalized_mitigation_train.csv', index=False)

# ==============================
# Scale validation and test sets
# ==============================

df_val = pd.read_csv(datapath+"encoded_Mitigation_validation.csv")
df_val[feature_numeric_cols] = scaler.transform(df_val[feature_numeric_cols])
df_val[feature_numeric_cols] = df_val[feature_numeric_cols].clip(lower=0, upper=1)
df_val.to_csv(datapath+'normalized_mitigation_validation.csv', index=False)

df_test = pd.read_csv(datapath+"encoded_mitigation_test.csv")
df_test[feature_numeric_cols] = scaler.transform(df_test[feature_numeric_cols])
df_test[feature_numeric_cols] = df_test[feature_numeric_cols].clip(lower=0, upper=1)
df_test.to_csv(datapath+'normalized_mitigation_test.csv', index=False)

print("Normalization completed and saved successfully.")


# del df_train, df_val, df_test

Normalization completed and saved successfully.


### Step 4 - Multi-Label Feature Selections

#### Multi-Label Feature Selection using CorrMI

In [19]:
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_selection import mutual_info_classif

# ============================
# Load datasets
# ============================
df_train = pd.read_csv(datapath+"normalized_mitigation_train.csv")

# ============================
# Define feature columns & labels
# ============================
LABELS = [f"M{i}" for i in range(1, 17)]   # M1–M16
ALL_NUMERIC_FEATURES = df_train.drop(['Attack_Type', 'Label'] + LABELS, axis=1).select_dtypes(include=[np.number]).columns.tolist()

# ============================
# Correlation filtering (global for all labels)
# ============================
corr_matrix = df_train[ALL_NUMERIC_FEATURES].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]

print(f"Correlation filtered: {len(high_corr_drop)} features removed")

X_filtered = df_train[ALL_NUMERIC_FEATURES].drop(columns=high_corr_drop)
filtered_feature_list = X_filtered.columns.tolist()

# ============================
# Mutual Information per label (M1...M16)
# ============================
top_features_per_label = {}
union_features = set()

for lbl in LABELS:
    print(f"\n=== Processing label: {lbl} ===")

    y = df_train[lbl]

    # Compute MI
    mi = mutual_info_classif(X_filtered, y, random_state=42)
    mi_df = pd.DataFrame({'Feature': filtered_feature_list, 'MI': mi})
    mi_df = mi_df.sort_values(by='MI', ascending=False)

    # Select top 20
    top20 = mi_df.head(20)['Feature'].tolist()
    top_features_per_label[lbl] = top20

    # Add to union set
    union_features.update(top20)

    print(f"Top 20 features for {lbl}: {top20}")

# ============================
# Save results
# ============================
joblib.dump(top_features_per_label, datapath+"mitigation_t20fpl_CorrMI.pkl")

joblib.dump(list(union_features), datapath+"mitigation_ufal_CorrMI.pkl")

print("\nSaved:")
print(" - Mitigation_t20fpl_CorrMI.pkl")
print(" - Mitigation_ufal_CorrMI.pkl")
print(f"\nUnion feature count: {len(union_features)}")

# ============================
# Save datasets using union feature set
# ============================
def save_corrmi_dataset(df, feature_list, out_path):
    df_out = df[list(feature_list) + LABELS + ['Label', 'Attack_Type']]
    df_out.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return df_out

df_train_final = save_corrmi_dataset(
    df_train, union_features,
    datapath+"CorrMI_mitigation_train.csv"
)

df_val = pd.read_csv(datapath+"normalized_mitigation_validation.csv")
df_test = pd.read_csv(datapath+"normalized_mitigation_test.csv")

df_val_final = save_corrmi_dataset(
    df_val, union_features,
    datapath+"CorrMI_mitigation_validation.csv"
)

df_test_final = save_corrmi_dataset(
    df_test, union_features,
    datapath+"CorrMI_mitigation_test.csv"
)

print("\nPreview train:")
print(df_train_final.head())


Correlation filtered: 35 features removed

=== Processing label: M1 ===
Top 20 features for M1: ['Init Fwd Win Byts', 'Down/Up Ratio', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Init Bwd Win Byts', 'Bwd Pkt Len Mean', 'Fwd PSH Flags', 'URG Flag Cnt', 'Fwd Pkt Len Min', 'Dst Port', 'Bwd Pkt Len Min', 'Tot Bwd Pkts', 'Flow Pkts/s', 'Pkt Len Var', 'FIN Flag Cnt', 'Tot Fwd Pkts', 'Bwd Pkts/s', 'TotLen Fwd Pkts', 'Src Port', 'Flow Byts/s']

=== Processing label: M2 ===
Top 20 features for M2: ['Init Fwd Win Byts', 'Down/Up Ratio', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Init Bwd Win Byts', 'Bwd Pkt Len Mean', 'Fwd PSH Flags', 'URG Flag Cnt', 'Fwd Pkt Len Min', 'Dst Port', 'Bwd Pkt Len Min', 'Tot Bwd Pkts', 'Flow Pkts/s', 'Pkt Len Var', 'FIN Flag Cnt', 'Tot Fwd Pkts', 'Bwd Pkts/s', 'TotLen Fwd Pkts', 'Src Port', 'Flow Byts/s']

=== Processing label: M3 ===
Top 20 features for M3: ['Init Fwd Win Byts', 'Down/Up Ratio', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Init Bwd Win Byts', 'Bwd Pkt Len Mean', 

#### Multi-Label Feature Selectin using Boruta

In [20]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import joblib

# =========================================================
# Load datasets
# =========================================================
df_train = pd.read_csv(datapath+"normalized_mitigation_train.csv")
df_val   = pd.read_csv(datapath+"normalized_mitigation_validation.csv")
df_test  = pd.read_csv(datapath+"normalized_mitigation_test.csv")

# Mitigation labels (multi-label)
LABELS = [f"M{i}" for i in range(1, 17)]

# All numeric features
X_all = df_train.drop(['Attack_Type', 'Label'] + LABELS, axis=1).select_dtypes(include=[np.number])
feature_cols = X_all.columns.tolist()

# =========================================================
# Boruta Feature Selection for each mitigation label
# =========================================================
top20_dict = {}
union_features = set()

for lbl in LABELS:
    print(f"\n========== Running Boruta for {lbl} ==========")

    y = df_train[lbl].values
    X = X_all.values

    # Random forest base estimator
    rf = RandomForestClassifier(
        n_jobs=-1, 
        max_depth=5, 
        random_state=42
    )

    # Boruta setup
    boruta = BorutaPy(
        estimator=rf,
        n_estimators='auto',
        random_state=42,
        verbose=0
    )

    # Fit Boruta
    boruta.fit(X, y)

    # Selected features
    selected_mask = boruta.support_
    selected_features = X_all.columns[selected_mask].tolist()
    print(f"Selected by Boruta ({lbl}): {len(selected_features)} features")

    # Rank using original RF importances
    importances = boruta.estimator.feature_importances_[:len(feature_cols)]
    ranking_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importances,
        'selected': selected_mask
    })

    ranking_df = ranking_df.sort_values(by='importance', ascending=False)

    # Top 20 features for this label
    top20 = ranking_df.head(20)['feature'].tolist()
    top20_dict[lbl] = top20

    # Add to union set
    union_features.update(top20)

    print(f"Top 20 for {lbl}: {top20}")

# =========================================================
# Save results
# =========================================================
joblib.dump(top20_dict,
    datapath+"Mitigation_t20fpl_boruta.pkl"
)

joblib.dump(list(union_features),
    datapath+"Mitigation_ufal_boruta.pkl"
)

print("\n========= Summary =========")
print(f"Saved: top20_boruta_per_label.pkl")
print(f"Saved: union_boruta_features.pkl")
print(f"Total unique union features: {len(union_features)}\n")

# =========================================================
# Build final datasets using UNION FEATURE SET
# =========================================================
def save_boruta_dataset(df, feature_set, out_path):
    df_out = df[list(feature_set) + LABELS + ['Label', 'Attack_Type']]
    df_out.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return df_out

df_train_final = save_boruta_dataset(df_train, union_features, datapath+"Boruta_mitigation_train.csv" )

save_boruta_dataset(df_val, union_features, datapath+"Boruta_mitigation_validation.csv")

save_boruta_dataset(df_test, union_features, datapath+"Boruta_mitigation_test.csv")

print("\nPreview:")
print(df_train_final.head())



========== Running Boruta for M1 ==========
Selected by Boruta (M1): 0 features
Top 20 for M1: ['Dst Port', 'PSH Flag Cnt', 'Pkt Size Avg', 'Down/Up Ratio', 'ECE Flag Cnt', 'CWE Flag Count', 'URG Flag Cnt', 'ACK Flag Cnt', 'RST Flag Cnt', 'Bwd Seg Size Avg', 'SYN Flag Cnt', 'FIN Flag Cnt', 'Pkt Len Var', 'Pkt Len Std', 'Pkt Len Mean', 'Pkt Len Max', 'Fwd Seg Size Avg', 'Subflow Fwd Pkts', 'Bwd Pkts/s', 'Active Std']

========== Running Boruta for M2 ==========
Selected by Boruta (M2): 0 features
Top 20 for M2: ['Dst Port', 'PSH Flag Cnt', 'Pkt Size Avg', 'Down/Up Ratio', 'ECE Flag Cnt', 'CWE Flag Count', 'URG Flag Cnt', 'ACK Flag Cnt', 'RST Flag Cnt', 'Bwd Seg Size Avg', 'SYN Flag Cnt', 'FIN Flag Cnt', 'Pkt Len Var', 'Pkt Len Std', 'Pkt Len Mean', 'Pkt Len Max', 'Fwd Seg Size Avg', 'Subflow Fwd Pkts', 'Bwd Pkts/s', 'Active Std']

========== Running Boruta for M3 ==========
Selected by Boruta (M3): 0 features
Top 20 for M3: ['Dst Port', 'PSH Flag Cnt', 'Pkt Size Avg', 'Down/Up Ratio', 